# Forecasting Electricity Price Volatility in Texas
### ERCOT Houston Hub — Erdos Institute Data Science Bootcamp 2026

---

**Team** | Spring 2026 Cohort  
**Goal** | Predict next-day RTM price volatility and flag price spikes at the Houston Hub

---
## 1. The Problem

**ERCOT** (Electric Reliability Council of Texas) runs a two-settlement electricity market:

| Market | When cleared | Price |
|---|---|---|
| Day-Ahead Market (DAM) | Day before delivery (~noon) | Predictable |
| Real-Time Market (RTM) | 15-min intervals, day-of | Highly volatile |

**The challenge:** RTM prices can deviate dramatically from DAM prices — sometimes reaching **$9,000/MWh** during extreme events (Winter Storm Uri, Feb 2021). These spikes create enormous financial risk for market participants.

**Our prediction task:**  
Using only information available at **midnight (12:05 AM CST)** before delivery day D, predict:
1. **How volatile** will hourly RTM prices be? → `log(RTM price std)` for each of the 24 hours of day D  
2. **Will there be a spike?** → binary flag when RTM mean > $100/MWh (~top 3% of hours)

**Why this matters:** Helps generators, load-serving entities, and traders hedge risk, dispatch assets efficiently, and plan reserves.

---
## 2. Data Sources

**Analysis window: July 2017 → December 2025** (~74,000 hourly observations)

### ERCOT Public Reports (9 datasets)

| Dataset | What it contains | Model role |
|---|---|---|
| NP6-905-CD | RTM settlement prices (15-min) | **Target variable** |
| NP4-190-CD | Day-Ahead Market clearing prices | Top predictor |
| NP4-523-CD | DAM system lambda | Congestion signal |
| NP4-188-CD | Ancillary service prices (RegUp, RRS, etc.) | Reserve market stress |
| NP6-346-CD | Actual load — Houston hub | Demand signal (48h lag) |
| NP4-732-CD | Wind generation actual + forecast | Renewable uncertainty |
| NP3-565-CD | Load forecast | Net load proxy |
| NP3-233-CD | Outage capacity | Supply-side stress |
| NP6-345-CD | Load by weather zone | Zone-level demand |

### Weather (Open-Meteo API)
4 Houston-specific features: temperature, humidity, wind gust, precipitation

### Feature engineering adds 8 derived features:
`fc_net_load`, `dam_rtm_spread`, `abs_dam_rtm_spread`, `week`, `load_lag7d`, `rtm_price_std_lag7d`, `rtm_price_mean_lag7d`, `outage_fraction`

**Selected model (XGBoost v2_XGB): 30 features** (29 base + `system_lambda`)  
_Note: `garch_cond_vol` is a 31st feature explored in v3 but excluded by CV — lagged RTM features already capture the volatility-clustering signal._

### Why XGBoost?

Many of our 31 features are **highly correlated** (e.g., DAM price and system lambda: r = 0.9996; 7-day lag features correlated with their same-day counterparts). This makes linear models fragile.

XGBoost handles correlated features naturally through tree splitting, captures **non-linear interactions** (e.g., wind error × load level), and is robust to outliers — important given the extreme spike distribution. Our walk-forward cross-validation confirmed it substantially outperforms all linear and time-series baselines:

| Model family | Best CV R² |
|---|---|
| ARIMA (univariate) | −0.490 |
| HAR-Ridge | 0.108 |
| Ridge (29 features) | 0.145 |
| HAR + Full-Ridge (32 features) | 0.220 |
| XGBoost v2_XGB (30 features, no GARCH) | 0.2827 |
| **XGBoost v3 (31 features, +GARCH)** | **0.2861** |


---
## 3. Target Variable

**Regression target:** `log(RTM price std + 1)` per hourly delivery slot  
→ Log transform stabilizes the heavy-tailed distribution and makes errors interpretable

**Classification target:** `spike_flag = 1` if RTM mean price > $100/MWh  
→ ~3% of hours (2,400 training positives), chosen at ~p97 of price distribution

The **5-point RTM std** includes the boundary interval from the next hour — Hour 23's std requires the midnight (00:00) interval, which posts at ~00:02 CST. This is why our prediction cutoff is **00:05 CST** (5 minutes after midnight), not exactly midnight.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})
PROC = Path('data/processed/ercot')

train = pd.read_parquet(PROC / 'train_features.parquet')
test  = pd.read_parquet(PROC / 'test_features.parquet')
all_df = pd.concat([train, test]).sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Monthly mean log-volatility
monthly = all_df['log_rtm_std'].resample('ME').mean()
axes[0].plot(monthly.index, monthly.values, color='steelblue', lw=1.5)
axes[0].axvspan(pd.Timestamp('2021-02-01'), pd.Timestamp('2021-03-01'),
                color='red', alpha=0.25, label='Winter Storm Uri')
axes[0].axvline(pd.Timestamp('2025-01-01'), color='orange', ls='--', lw=1.5, label='Train/Test split')
axes[0].set_title('Monthly Mean Log-Volatility (Houston RTM)')
axes[0].set_ylabel('log(RTM std + 1)')
axes[0].legend(fontsize=9)

# Spike distribution
spike_by_month = all_df['spike_flag'].resample('ME').mean() * 100
axes[1].bar(spike_by_month.index, spike_by_month.values, width=25, color='crimson', alpha=0.7)
axes[1].set_title('Monthly Spike Rate (RTM > $100/MWh)')
axes[1].set_ylabel('% of hours')
axes[1].axhline(all_df['spike_flag'].mean() * 100, color='black', ls='--', lw=1, label=f'Overall: {all_df["spike_flag"].mean()*100:.1f}%')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('figures/modeling/pres_target.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Training hours: {len(train):,}  |  Test hours: {len(test):,}")
print(f"Spike rate (train): {train['spike_flag'].mean()*100:.1f}%  |  Spike rate (test): {test['spike_flag'].mean()*100:.1f}%")

---
## 4. Methodology

### Train / Test / Hold-out Split

| Split | Dates | Hours | Purpose |
|---|---|---|---|
| **Train** | 2017-07-04 → 2024-12-31 | 65,712 | Model fitting & CV |
| **Test** | 2025-01-01 → 2025-12-31 | 8,760 | Final evaluation — never used for selection |
| **Hold-out** | 2026 | — | Reserved — not touched |

### Prediction Window

At **00:05 CST on delivery day D**, we predict all 24 hours of day D using:
- DAM prices (posted ~12:35 CST on D-1) ✓ available  
- RTM prices from day D-1 (last interval posts ~00:02 CST D) ✓ available  
- Load actuals from D-2 (post ~05:50 CST D, after cutoff) — 48h lag  
- Wind actuals from D-2 (post ~00:31 CST D, after cutoff) — 48h lag  

### Walk-Forward Cross-Validation

Expanding training window — **no random splits**, no shuffling:

| Fold | Train window | Validation year |
|---|---|---|
| 1 | 2017–2021 | 2022 |
| 2 | 2017–2022 | 2023 |
| 3 | 2017–2023 | 2024 |

Test 2025 is **never touched** during model development — used only for final reporting.

### Post-Uri Training Window

Winter Storm Uri (Feb 2021) caused a permanent regime shift in ERCOT volatility dynamics. Training on 2021–2024 only (post-Uri) gives slightly better test performance than using full history (R²=0.382 vs 0.378), confirming the structural break.

---
## 5. Model Leaderboard

Models are compared by **walk-forward CV R²** (val folds 2022/2023/2024 — never touching the test set). The test set (2025) is used only for final reporting on the selected model.

### Volatility Regression — CV Comparison (target: log RTM std)

| Model | Features | CV R² | Test R² | Selection |
|---|---|---|---|---|
| ARIMA (univariate) | 1 | −0.490 | — | ✗ |
| HAR-Ridge | 3 | 0.108 | 0.119 | ✗ |
| Full-Ridge | 29 | 0.145 | 0.167 | ✗ |
| HAR+Full-Ridge (best linear) | 32 | 0.220 | 0.217 | ✗ |
| XGBoost v2_XGB (no GARCH) | 30 | 0.283 | 0.375 | ✗ |
| XGBoost v3 +GARCH (original params) | 31 | 0.286 | 0.384 | ✗ |
| Ensemble (Ridge + XGB v3, α≈0.72) | 29+31 | 0.309 | 0.381 | ✗ α tuned for weaker model |
| **XGBoost v3 tuned +GARCH** | **31** | **0.311** | **0.399** | **✅ FINAL** |

**Final model: XGBoost v3 tuned (post-Uri 2021–2024 window)**
- Hyperparams: `max_depth=4, lr=0.03, n_estimators=600` (from grid search over 54 combos)
- Test R²=**0.399**, RMSE=**0.662** — reported once on held-out 2025 data, never used for selection
- Ensemble adds no value over the tuned model (α calibrated for weaker original XGB)

In [ ]:
import pickle
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Model leaderboard bar chart ──
models = [
    ('HAR-Ridge\n(3 TS lags)',    None,  0.108, 'lightgray'),
    ('Ridge\n(29 feat)',           None,  0.145, 'lightgray'),
    ('XGBoost v2_XGB\n(30 feat)', 0.375, 0.2827, 'steelblue'),
    ('XGBoost v3\n(31 feat)',      0.378, 0.2861, 'steelblue'),
    ('XGBoost v3 best\n(2021–24)', 0.384, None,  'crimson'),
]
names = [m[0] for m in models]
test_r2 = [m[1] if m[1] else 0 for m in models]
cv_r2   = [m[2] if m[2] else 0 for m in models]
colors  = [m[3] for m in models]

x = np.arange(len(names))
w = 0.35
axes[0].bar(x - w/2, cv_r2, w, label='CV R² (mean)', color='lightblue', edgecolor='gray')
axes[0].bar(x + w/2, test_r2, w, label='Test R² (2025)', color=colors, edgecolor='gray')
axes[0].set_xticks(x); axes[0].set_xticklabels(names, fontsize=9)
axes[0].set_ylabel('R²')
axes[0].set_title('Model Comparison — Volatility Regression')
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 0.50)
axes[0].axhline(0.384, color='crimson', ls='--', lw=1, alpha=0.6)

# ── Right: Feature importance (top 10) ──
with open(PROC / 'model_xgb_reg_v3_best.pkl', 'rb') as f:
    m_best = pickle.load(f)

feat_names = list(m_best.get_booster().feature_names)
importances = m_best.feature_importances_
fi = sorted(zip(feat_names, importances), key=lambda x: x[1], reverse=True)[:10]
fi_names = [x[0] for x in fi]
fi_vals  = [x[1]*100 for x in fi]

axes[1].barh(range(len(fi_names)), fi_vals[::-1], color='steelblue', alpha=0.8)
axes[1].set_yticks(range(len(fi_names)))
axes[1].set_yticklabels(fi_names[::-1], fontsize=9)
axes[1].set_xlabel('Feature Importance (%)')
axes[1].set_title('Top 10 Features — XGBoost v3 Best')

plt.tight_layout()
plt.savefig('figures/modeling/pres_leaderboard.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 6. Key Findings

1. **Final model: XGBoost v3 tuned (31 features, +GARCH)** — selected by walk-forward CV (mean R²=0.311). Hyperparameters tuned by grid search (`max_depth=4, lr=0.03`), trained on post-Uri window 2021–2024. Test R²=**0.399**, RMSE=**0.662** on 2025 holdout (reported once).

2. **DAM price is the dominant predictor** (26% importance) — the market already prices in most known information by noon the day before.

3. **GARCH helps — CV confirms inclusion** — GARCH(1,1)-t conditional volatility was added as a 31st feature (v3). Walk-forward CV shows GARCH provides a small but consistent improvement (Δ=+0.003, v3 CV R²=0.286 vs v2_XGB 0.283). The GARCH exercise also confirmed **IGARCH dynamics** (α+β=1.0) — a genuine finding about ERCOT market structure: volatility shocks never decay.

4. **Volatility is persistent** — yesterday's RTM std (`rtm_std_lag24`) and reserve market tightness (`mcpc_regup`) are the next strongest signals after DAM price.

5. **Post-Uri regime shift** — training only on 2021–2024 (post-Uri) beats full 2017–2024 history (R²=0.399 vs 0.385), confirming that Winter Storm Uri permanently altered ERCOT volatility dynamics.

6. **Hyperparameter tuning pays off** — grid search (54 combos) found shallower trees (`max_depth=4`) and slower learning (`lr=0.03`) improve both CV R² (+0.025) and test R² (+0.015) over the default configuration.

7. **Ensemble doesn't add value** — blending Ridge (CV R²=0.145) with tuned XGBoost at α=0.72 gives test R²=0.381, worse than tuned XGB alone (0.399). The linear model is too weak to contribute useful diversity.

8. **Model drift is real** — rolling 4-year window retraining (Option C) beats a fixed model in 45/53 weeks; 6 structural change points detected in 2025. ERCOT markets evolve, and models need periodic retraining.

9. **Spike detection is hard** — 2.2% spike rate, AUC=0.904 (good discrimination) but spikes still show 2.8× higher MAE than normal hours.

---

### Limitations

- **R²=0.399** — about 60% of log-volatility variance is unexplained. Much of that is genuinely unpredictable from day-ahead data.
- **Classifier calibration** — spike probabilities are inflated by class-imbalance weighting; AUC/F1 are the operative metrics, not Brier score.
- **2026 hold-out** — performance on 2026 data has not been evaluated (preserved as a clean hold-out).
- **No fundamentals data** — fuel prices, transmission constraints, and generation capacity bids are not used; incorporating them could improve predictions.

In [ ]:
# Error analysis: MAE by hour and spike vs non-spike
from sklearn.metrics import r2_score, mean_squared_error

# Rebuild needed engineered features for prediction
def add_eng(df):
    df = df.copy()
    df['fc_net_load']         = df['fc_coast'] - df['wf_stwpf_lz_south_houston']
    df['dam_rtm_spread']      = df['dam_price_houston'] - df['rtm_mean_lag24']
    df['abs_dam_rtm_spread']  = df['dam_rtm_spread'].abs()
    df['week']                = df.index.isocalendar().week.astype(int)
    df['load_lag7d']             = df['load_houston_lag48'].shift(168)
    df['rtm_price_std_lag7d']    = df['rtm_std_lag24'].shift(168)
    df['rtm_price_mean_lag7d']   = df['rtm_mean_lag24'].shift(168)
    df['outage_fraction']     = df['total_resource_mw'] / (df['fc_system_total'] + 1)
    return df

all_eng = add_eng(all_df)
test_eng = all_eng[all_eng.index >= '2025-01-01'].copy()

feat_names = list(m_best.get_booster().feature_names)
# garch_cond_vol not available here; fill with 0 (feature importance ~1%)
test_eng['garch_cond_vol'] = 0.0
X_te = test_eng[feat_names].fillna(0)
y_te = test_eng['log_rtm_std']
pred = m_best.predict(X_te)
abs_err = np.abs(y_te.values - pred)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# MAE by hour
by_hour = pd.Series(abs_err, index=test_eng.index).groupby(test_eng.index.hour).mean()
axes[0].bar(by_hour.index, by_hour.values, color='steelblue', alpha=0.8)
axes[0].axhline(abs_err.mean(), color='red', ls='--', lw=1.5, label=f'Mean MAE={abs_err.mean():.3f}')
axes[0].set_xlabel('Hour of day (CST)'); axes[0].set_ylabel('MAE (log scale)')
axes[0].set_title('MAE by Hour of Day')
axes[0].legend(fontsize=9)

# MAE by month
by_month = pd.Series(abs_err, index=test_eng.index).groupby(test_eng.index.month).mean()
axes[1].bar(by_month.index, by_month.values, color='coral', alpha=0.8)
axes[1].axhline(abs_err.mean(), color='red', ls='--', lw=1.5)
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
axes[1].set_title('MAE by Month (2025 test)')
axes[1].set_ylabel('MAE (log scale)')

# Spike vs non-spike
spike_mask = test_eng['spike_flag'] == 1
mae_s = abs_err[spike_mask.values].mean()
mae_n = abs_err[~spike_mask.values].mean()
axes[2].bar(['Non-spike\n(RTM ≤$100)', 'Spike\n(RTM >$100)'],
            [mae_n, mae_s], color=['steelblue', 'crimson'], alpha=0.85)
axes[2].set_title(f'MAE: Spike vs Non-Spike (ratio={mae_s/mae_n:.1f}×)')
axes[2].set_ylabel('MAE (log scale)')
for i, v in enumerate([mae_n, mae_s]):
    axes[2].text(i, v+0.01, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/modeling/pres_error.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Conclusion

**We built a two-output forecasting system for ERCOT Houston Hub:**

> *Given all market information available at midnight, predict tomorrow's hourly price volatility and flag hours likely to spike above $100/MWh.*

**Final model: XGBoost v3 tuned (31 features, +GARCH)** — selected by walk-forward CV (mean CV R²=0.311)
- Hyperparameters: `max_depth=4, lr=0.03, n_estimators=600` (grid search over 54 combinations)
- Training window: post-Uri 2021–2024 (regime shift confirmed)
- Volatility regression: **Test R² = 0.399**, RMSE = 0.662 on held-out 2025 data (reported once)
- Spike detection: **AUC = 0.904**, F1 = 0.510

**Key takeaways:**
- DAM price is the single most informative predictor — the market is efficient at pricing known risks
- GARCH conditional volatility provides a measurable CV improvement and is included in the final model; IGARCH dynamics confirmed as a structural finding about ERCOT
- Hyperparameter tuning adds a meaningful boost: shallower trees + slower learning rate outperform defaults
- Ensemble blending doesn't help — the linear model is too weak to contribute useful diversity
- Regime shifts (post-Uri) and market drift mean static models degrade; **periodic retraining is necessary** (Option C beats fixed model in 45/53 weeks, 6 change points detected)
- Extreme spikes remain hard to forecast — they arise from combinations of factors outside historical patterns

**Next steps:**
- Evaluate on 2026 hold-out once model is locked
- Incorporate fuel prices and transmission constraint data
- Explore online/streaming model updates as real-time data arrives